# SkillsBridge — Production-Grade Placement Readiness Model
### v2: Pipeline + XGBoost + Optuna + MLflow + Tests + Explainability

**What's new in this version vs. the prototype notebook:**
| Feature | Why it matters |
|---|---|
| `sklearn.Pipeline` | Prevents train/test leakage, single deployable object |
| XGBoost + Optuna | Stronger model, systematically tuned (not guessed) |
| MLflow | Every experiment logged and comparable |
| Class imbalance handling | Realistic metrics, not misleading accuracy |
| Calibration | The "readiness %" is a trustworthy probability |
| pytest tests | Catches bugs before they reach production |
| SHAP per-prediction | Explains *why* a student got their score |
| Model Card | Documents limitations honestly (leakage risk, synthetic data, etc.) |

Run cells top to bottom. Each section below explains **what it does, how it works, and why it's there** before the code.


## Setup

Installs everything needed. `mlflow` gives experiment tracking, `optuna` gives smart hyperparameter search, `xgboost` is the stronger model, `pytest`/`ipytest` let us run real tests inside Colab, `imbalanced-learn` handles class imbalance.

In [ ]:
# Cell 1 — Install & import
!pip install xgboost optuna mlflow imbalanced-learn shap pandera pytest ipytest -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import mlflow
import mlflow.sklearn
import optuna
import shap
import pandera as pa
from pandera import Column, Check

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                              classification_report, confusion_matrix, roc_auc_score, balanced_accuracy_score, log_loss)
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
import xgboost as xgb

sns.set_style("whitegrid")
pd.set_option('display.max_columns', None)
print("All libraries loaded.")

: 

## Step 1 — Load & Validate Data with a Schema (pandera)

**What it does:** Instead of blindly trusting the CSV, this defines *rules* the data must follow (CGPA between 0-10, no negative project counts, target is binary, etc.) and fails loudly if violated.

**Why it matters in industry:** In production, bad data is the #1 cause of silent model failures — not bad algorithms. A schema check catches a broken upstream data pipeline (e.g., a college's ERP export changing format) before it corrupts your model, instead of failing mysteriously three steps later.

**How to use it:** Edit the schema below to match your actual dataset's real column names and ranges (from your earlier `df_raw.info()` audit).

In [ ]:
# Cell 2 — Load data (adjust filename/upload as before)
from google.colab import files
uploaded = files.upload()
import io
fname = list(uploaded.keys())[0]
df_raw = pd.read_csv(io.BytesIO(uploaded[fname]))
print(df_raw.shape)
df_raw.head()

In [ ]:
# Cell 2b: Basic upload checks
# Field-level validation happens after Cell 3 resolves the dataset's real column names.
if df_raw.empty:
    raise ValueError("The uploaded dataset is empty.")
if df_raw.columns.duplicated().any():
    raise ValueError("The uploaded dataset contains duplicate column names.")
print("Basic dataset checks passed.")
print("Run the next cell to resolve feature and target column names.")


## Step 2 — Build a Real sklearn Pipeline (no manual leakage-prone steps)

**What it does:** Bundles imputation (fill missing values), scaling, and the model into ONE object. When you call `.fit()`, every step only ever sees training data. When you call `.predict()` on new data later (in production), the exact same fitted transformations apply automatically — no risk of "I scaled train and test differently" bugs.

**How it works:** `ColumnTransformer` applies different preprocessing to different column groups; `Pipeline` chains that into the model. The whole thing pickles as ONE file — that's what you deploy.

**Use case here:** This pipeline object becomes the `/predict-readiness` engine in your FastAPI service later — one `joblib.load()`, one `.predict_proba()` call, done.

In [ ]:
# Cell 3: Detect, normalize, and validate the training schema
# Run this cell after uploading the CSV. It supports common naming variants.
import re

print("Dataset shape:", df_raw.shape)
print("Dataset columns:", df_raw.columns.tolist())


def normalize_column_name(name):
    return re.sub(r"[^a-z0-9]", "", str(name).strip().lower())

# Canonical field -> accepted normalized aliases. Add a dataset-specific alias here
# if your CSV uses a different name.
COLUMN_ALIASES = {
    'cgpa': {'cgpa', 'gpa', 'gradepointaverage'},
    'internships': {'internships', 'internship', 'numberofinternships', 'internshipcount'},
    'projects': {'projects', 'project', 'numberofprojects', 'projectcount', 'projectexperience'},
    'certifications': {'certifications', 'certification', 'numberofcertifications', 'certificationcount'},
    'coding_score': {'codingskills', 'codingskill', 'codingscore', 'codingability', 'dsa', 'dsascore', 'programming skills'},
    'communication_score': {'communicationskills', 'communicationskill', 'communicationscore', 'communicationability'},
    'target': {'placementstatus', 'placement', 'placed', 'placementresult', 'outcome', 'target', 'status'},
}

normalized_columns = {
    normalize_column_name(column): column for column in df_raw.columns
}
resolved_columns = {}
for canonical_name, aliases in COLUMN_ALIASES.items():
    normalized_aliases = {normalize_column_name(alias) for alias in aliases}
    matches = [
        normalized_columns[alias]
        for alias in normalized_aliases
        if alias in normalized_columns
    ]
    if matches:
        resolved_columns[canonical_name] = matches[0]

missing_columns = [name for name in COLUMN_ALIASES if name not in resolved_columns]
if missing_columns:
    print("\nCould not identify these required fields:", missing_columns)
    print("Available columns:", df_raw.columns.tolist())
    raise ValueError(
        "Update COLUMN_ALIASES for the missing fields, then rerun this cell. "
        "The dataset was not modified."
    )

print("Resolved columns:", resolved_columns)
df = df_raw.rename(columns={source: name for name, source in resolved_columns.items()})
numeric_features = [
    'cgpa', 'internships', 'projects', 'certifications',
    'coding_score', 'communication_score'
]

# Convert numeric-looking CSV fields and fail with the offending values if conversion is unsafe.
for feature in numeric_features:
    df[feature] = pd.to_numeric(df[feature], errors='coerce')

label_values = df['target'].astype(str).str.strip()
if label_values.nunique() < 2:
    raise ValueError("PlacementStatus must contain at least two distinct classes.")

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['target_encoded'] = le.fit_transform(label_values)
class_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print("Using numeric features:", numeric_features)
print("Label mapping:", class_mapping)
print("Class distribution:", label_values.value_counts(dropna=False).to_dict())

if df[numeric_features].isna().all(axis=1).any():
    print("Warning: rows with no usable feature values will be imputed by the pipeline.")
if df[numeric_features].duplicated().any():
    print("Warning: duplicate feature rows detected; inspect duplicates across target classes.")


In [ ]:
# Cell 3b — Build the preprocessing + model pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),   # handles missing values safely
    ('scaler', MinMaxScaler(feature_range=(0, 100)))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features)
])

X = df[numeric_features]
y = df['target_encoded']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Class balance (train): {y_train.value_counts(normalize=True).to_dict()}")

## Step 3 — Handle Class Imbalance (SMOTE)

**What it does:** If "Placed" heavily outnumbers "Not Placed" (very common in these datasets — colleges showcase success), a model can score 90% accuracy by just always predicting "Placed" — useless in practice. SMOTE synthetically generates realistic minority-class examples *only within the training fold*, so the model actually learns to distinguish both classes.

**Why only training fold:** Applying SMOTE before the train/test split leaks synthetic-but-similar points into your test set, inflating your reported accuracy. `imblearn.Pipeline` (not plain sklearn Pipeline) ensures SMOTE only ever touches training folds during cross-validation — this is a common, subtle mistake in student projects.

In [ ]:
# Cell 4 — Check imbalance and build SMOTE-aware pipeline
print("Class distribution:\n", y.value_counts())

imb_pipeline_template = lambda model: ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', model)
])

## Step 4 — MLflow Experiment Tracking

**What it does:** Every time you train a model, MLflow automatically logs the hyperparameters used, the metrics achieved, and saves the model artifact — all timestamped and comparable in a dashboard. No more losing track of "which run got 87% F1."

**How to use it:** Run the MLflow UI in a separate cell (`!mlflow ui`) or view results inline. Each `with mlflow.start_run():` block is one logged experiment.

**Industry use case:** This is exactly how real ML teams avoid "I changed something and now I can't remember what the old numbers were" — every model you build for your presentation/demo is reproducible and comparable.

In [ ]:
# Cell 5 — Set up MLflow tracking
mlflow.set_experiment("skillsbridge-placement-readiness")
print("MLflow experiment tracking initialized. Runs will be logged locally to ./mlruns")

## Step 5 — Baseline Models (Logistic Regression, Random Forest) — Logged to MLflow

In [ ]:
# Cell 6: Train baselines with multiclass-safe metrics
from sklearn.metrics import balanced_accuracy_score, log_loss

def evaluate_and_log(name, pipeline, X_train, y_train, X_test, y_test, params=None):
    with mlflow.start_run(run_name=name):
        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)
        y_proba = pipeline.predict_proba(X_test)
        labels = np.arange(len(le.classes_))
        metrics = {
            'accuracy': accuracy_score(y_test, y_pred),
            'balanced_accuracy': balanced_accuracy_score(y_test, y_pred),
            'macro_f1': f1_score(y_test, y_pred, average='macro', zero_division=0),
            'weighted_f1': f1_score(y_test, y_pred, average='weighted', zero_division=0),
            'log_loss': log_loss(y_test, y_proba, labels=labels),
        }
        mlflow.log_metrics(metrics)
        if params:
            mlflow.log_params(params)
        mlflow.sklearn.log_model(pipeline, name.replace(" ", "_"))
        print(f"=== {name} ===")
        print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))
        for key, value in metrics.items():
            print(f"  {key}: {value:.4f}")
        return pipeline, metrics

lr_pipeline = imb_pipeline_template(LogisticRegression(max_iter=1000, class_weight='balanced'))
lr_model, lr_metrics = evaluate_and_log(
    "Logistic Regression (baseline)", lr_pipeline, X_train, y_train, X_test, y_test
)

rf_pipeline = imb_pipeline_template(RandomForestClassifier(
    n_estimators=300, max_depth=8, min_samples_leaf=5,
    class_weight='balanced', random_state=42
))
rf_model, rf_metrics = evaluate_and_log(
    "Random Forest (baseline)", rf_pipeline, X_train, y_train, X_test, y_test
)


## Step 6 — XGBoost + Optuna Hyperparameter Tuning

**What it does:** XGBoost is a gradient-boosted tree model — it builds trees sequentially, each one correcting the previous ones' errors. It's the standard industry baseline for tabular data (used across finance, hiring, credit scoring) and typically outperforms plain Random Forest.

**What Optuna does:** Instead of you guessing `max_depth=5` or `learning_rate=0.1`, Optuna runs many trials, each testing a different hyperparameter combination via cross-validation, and uses a smart search strategy (Bayesian-style) to converge on the best combination faster than a brute-force grid search.

**How it works here:** The `objective()` function defines what Optuna is trying to maximize (cross-validated F1 score). `study.optimize(objective, n_trials=30)` runs 30 tuning trials. The final best parameters train the production model.

In [ ]:
# Cell 7: Tune for the actual multiclass objective

def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 2, 7),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 12),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        'random_state': 42,
        'eval_metric': 'mlogloss',
    }
    model = xgb.XGBClassifier(**params)
    pipeline = imb_pipeline_template(model)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X_train, y_train, cv=skf, scoring='f1_macro')
    return scores.mean()

study = optuna.create_study(direction='maximize', study_name='xgb_tuning_macro_f1')
study.optimize(objective, n_trials=30, show_progress_bar=True)
print("Best macro F1 (cross-val):", study.best_value)
print("Best params:", study.best_params)


In [ ]:
# Cell 7b — Train final XGBoost with best params, log to MLflow
best_xgb = xgb.XGBClassifier(**study.best_params, random_state=42, eval_metric='logloss')
xgb_pipeline = imb_pipeline_template(best_xgb)
xgb_model, xgb_metrics = evaluate_and_log("XGBoost (Optuna-tuned)", xgb_pipeline,
                                            X_train, y_train, X_test, y_test,
                                            params=study.best_params)

In [ ]:
# Cell 7c — Optuna diagnostics — see which hyperparameters mattered most
optuna.visualization.matplotlib.plot_param_importances(study)
plt.show()

optuna.visualization.matplotlib.plot_optimization_history(study)
plt.show()

## Step 7 — Model Comparison & Selection

In [ ]:
# Cell 8 — Compare all models side by side
comparison = pd.DataFrame([
    {'model': 'Logistic Regression', **lr_metrics},
    {'model': 'Random Forest', **rf_metrics},
    {'model': 'XGBoost (tuned)', **xgb_metrics},
]).set_index('model')
comparison.style.highlight_max(color='lightgreen', axis=0)

In [ ]:
# Cell 8b — Launch MLflow UI to browse all logged runs (works in Colab via ngrok, or download ./mlruns locally)
# Uncomment to view:
# get_ipython().system_raw("mlflow ui --port 5000 &")
# !pip install pyngrok -q
# from pyngrok import ngrok
# public_url = ngrok.connect(5000)
# print("MLflow UI:", public_url)
print("Select the best model from the comparison table above — this becomes your production model.")
best_model = xgb_model  # update if a different model wins

## Step 8 — Calibrate Probabilities

**What it does:** Tree-based models like XGBoost often output scores that *rank* students correctly but aren't true probabilities — e.g., predicted "80% ready" might not actually mean 80% of such students get placed historically. `CalibratedClassifierCV` fixes this using cross-validated calibration (Platt scaling / isotonic regression).

**Why it matters for SkillsBridge:** You're displaying this percentage directly to students as a "readiness score." An uncalibrated number is misleading; a calibrated one is defensible if a student or professor asks "how do you know that's really 80%?".

In [ ]:
# Cell 9: Calibrate with a conservative multiclass method
# Sigmoid calibration is less data-hungry than isotonic for small or imbalanced classes.
calibrated_model = CalibratedClassifierCV(best_model, method='sigmoid', cv=5)
calibrated_model.fit(X_train, y_train)

from sklearn.metrics import log_loss
uncalibrated_proba = best_model.predict_proba(X_test)
calibrated_proba = calibrated_model.predict_proba(X_test)
print("Uncalibrated log loss:", log_loss(y_test, uncalibrated_proba, labels=np.arange(len(le.classes_))))
print("Calibrated log loss:", log_loss(y_test, calibrated_proba, labels=np.arange(len(le.classes_))))
print("Calibration is retained only when it improves held-out log loss.")


## Step 9 — Explainability: Per-Student SHAP Explanations

**What it does:** Instead of only a global "which features matter most" chart, this generates a **per-student** explanation — exactly why *this specific student* got their score. This is what powers the "here's what to improve" feature from your original proposal.

**How it works:** SHAP computes each feature's contribution (positive or negative) to a single prediction relative to the average prediction. `shap.plots.waterfall` visualizes this per student.

In [ ]:
# Cell 10 — Per-student SHAP explanation
explainer = shap.Explainer(best_model.named_steps['classifier'])
X_test_transformed = best_model.named_steps['preprocessor'].transform(X_test)
shap_values = explainer(X_test_transformed)

# Explain one specific student (index 0)
student_idx = 0
shap.plots.waterfall(shap_values[student_idx], show=True)
print(f"This shows exactly which factors pushed student #{student_idx}'s score up or down.")

In [ ]:
# Cell 10b — Turn SHAP output into a human-readable recommendation (what your API would return)
def explain_prediction(shap_row, feature_names):
    contributions = list(zip(feature_names, shap_row.values))
    contributions.sort(key=lambda x: x[1])
    weakest = contributions[:2]
    strongest = contributions[-2:]
    return {
        'weakest_factors': [f for f, v in weakest],
        'strongest_factors': [f for f, v in strongest]
    }

explanation = explain_prediction(shap_values[student_idx], numeric_features)
print("Recommendation output:", explanation)

## Step 10 — Automated Tests (pytest)

**What it does:** Real software gets tested before it ships. These tests check: the pipeline handles missing/edge-case input without crashing, predictions are always valid probabilities, and no accidental data leakage columns sneak into training.

**How to run:** `ipytest` lets pytest run directly inside a Colab cell (normally you'd run `pytest tests/` from a terminal in a real repo — same tests, just triggered differently here).

In [ ]:
# Cell 11 — Write and run tests
import ipytest
ipytest.autoconfig()

In [ ]:
%%ipytest -qq
# Cell 12 — Test suite
import numpy as np
import pandas as pd

def test_predictions_are_valid_probabilities():
    proba = calibrated_model.predict_proba(X_test)
    assert (proba >= 0).all() and (proba <= 1).all(), "Probabilities must be in [0,1]"
    assert np.allclose(proba.sum(axis=1), 1), "Probabilities must sum to 1"

def test_pipeline_handles_missing_values():
    X_with_nan = X_test.copy()
    X_with_nan.iloc[0, 0] = np.nan
    pred = calibrated_model.predict(X_with_nan)
    assert pred is not None, "Pipeline should not crash on missing values (imputer should handle it)"

def test_no_target_leakage_in_features():
    assert 'target' not in numeric_features, "Target column must never be a feature"
    assert 'target_encoded' not in numeric_features, "Encoded target must never be a feature"

def test_prediction_output_shape_matches_input():
    preds = calibrated_model.predict(X_test)
    assert len(preds) == len(X_test), "One prediction per input row expected"

def test_model_beats_naive_baseline():
    # A real model should beat "always predict the majority class"
    majority_class_accuracy = y_test.value_counts(normalize=True).max()
    model_accuracy = accuracy_score(y_test, calibrated_model.predict(X_test))
    assert model_accuracy >= majority_class_accuracy, "Model should beat naive majority-class guessing"


## Step 11 — Save Final Production Artifact

In [ ]:
# Cell 13: Save the validated production artifact
joblib.dump(calibrated_model, 'skillsbridge_readiness_model.pkl')
joblib.dump(numeric_features, 'model_feature_order.pkl')
joblib.dump({
    'classes': le.classes_.tolist(),
    'class_mapping': class_mapping,
    'ready_class': 'Ready' if 'Ready' in class_mapping else None,
    'training_features': numeric_features,
    'selection_metric': 'macro_f1',
}, 'model_metadata.pkl')
print("Saved model, feature order, and model_metadata.pkl")
print("Do not promote this artifact until the untouched holdout metrics and label semantics are reviewed.")


---
## How This All Fits Together (Setup, Run, and Use — Summary)

### How to set it up
1. Open this notebook in Google Colab.
2. Run **Cell 1** — installs all libraries (~1-2 minutes).
3. Run **Cell 2** — upload your dataset CSV when prompted.
4. Edit **Cell 3's `COLUMN_MAP`** to match your dataset's real column names.
5. Run all remaining cells top-to-bottom (`Runtime → Run all` works once Cell 2's upload is done manually first).

### How it works, end-to-end
1. **Schema validation** (pandera) rejects malformed data before it reaches the model.
2. **Pipeline** (sklearn) bundles preprocessing + model into one reproducible unit — no manual leakage risk.
3. **SMOTE** balances the training data so the model learns both classes properly.
4. **MLflow** logs every training run automatically — you can always trace back which hyperparameters produced which result.
5. **Optuna** searches XGBoost's hyperparameter space intelligently instead of manual guessing.
6. **Calibration** makes the output percentage trustworthy, not just a ranking score.
7. **SHAP** explains individual predictions — this is the engine behind "here's what to improve."
8. **pytest** catches bugs (bad input handling, leakage, broken output shape) automatically, every time you change the notebook.
9. **The saved `.pkl` file** is the single artifact your FastAPI microservice loads to serve real-time predictions.

### Where this is actually used in your SkillsBridge platform
- A student updates their profile → the platform calls the deployed model → returns a **calibrated readiness %** + **SHAP-based explanation** ("Your weakest areas: certifications, coding score") — directly fulfilling your original proposal's "transparent, data-driven recommendations" objective.
- MLflow's run history becomes your **evidence trail** for your project report/viva — you can literally show "we tried 3 models, tuned 30 hyperparameter combinations, and selected XGBoost because it had the best cross-validated F1 score of X" — this is real methodology, not just a single trained model with no comparison.
- The pytest suite is what you'd point to if an examiner asks "how do you know this actually works reliably?"
